In [ ]:
# ==========================================
# Célula 1 - Configurações, Importações e Setup
# ==========================================
# Todas as importações e parâmetros globais do pipeline ficam AQUI (as demais células não reimportam).
import os
import re
import gc
import time
import glob
import shutil
import stat
import html
import datetime as dt
import openpyxl
import pandas as pd
from datetime import datetime, timedelta

# Selenium e Webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Janela de extração: do 1º dia do mês de HOJE até AGORA (nada do dia de hoje é descartado)
hoje = datetime.now()
primeiro_dia = hoje.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
pasta_mes = hoje.strftime("%Y%m")

data_de_str = primeiro_dia.strftime("%d/%m/%Y")   # campos de data dos relatórios antigos (Tickets)
data_ate_str = hoje.strftime("%d/%m/%Y")
data_de_iso = primeiro_dia.strftime("%Y-%m-%d")   # campos <input type="date"> (Enquetes)
data_ate_iso = hoje.strftime("%Y-%m-%d")
data_de_keys = primeiro_dia.strftime("%d%m%Y")    # digitação nativa via teclado
data_ate_keys = hoje.strftime("%d%m%Y")

dt_inicio_filtro = pd.Timestamp(primeiro_dia)     # guilhotina: tudo >= 1º dia do mês é substituído pelo Gold

# URLs do MKTZap
URL_MKTZAP = "https://EMPRESA.mktzap.com.br"
URL_LOGIN = f"{URL_MKTZAP}/login"
URL_CHAT = f"{URL_MKTZAP}/chat"
URL_ANALITICO = f"{URL_MKTZAP}/reports/analytic"
URL_ENQUETES = f"{URL_MKTZAP}/new/reports/polls"
URL_TICKETS = f"{URL_MKTZAP}/reports/history-tickets"
URL_BACKGROUND = f"{URL_MKTZAP}/new/reports/background-reports"

caminho_base = os.getcwd()

# Mapeamento Multi-Tenant
def montar_tenant(id_tenant, nome_exibicao, nome_empresa_mktzap):
    return {
        "id": id_tenant,
        "nome_exibicao": nome_exibicao,
        "nome_empresa_mktzap": nome_empresa_mktzap,
        "pasta_destino": os.path.join(caminho_base, "documentos", pasta_mes, nome_exibicao),
        "pasta_temp": os.path.join(caminho_base, f"downloads_temp_{id_tenant}"),
    }

tenants = [
    montar_tenant("A", "Tenant A", "Tenant A - Empresa"),
    montar_tenant("B", "Tenant B", "Tenant B - Empresa"),
    montar_tenant("C", "Tenant C", "Tenant C - Empresa"),
]

# Garantia da existência das pastas físicas
for t in tenants:
    os.makedirs(t["pasta_temp"], exist_ok=True)
    os.makedirs(t["pasta_destino"], exist_ok=True)

# Estrutura Global para Rastreamento de Erros de Extração
registro_execucao = []

print(f"✅ Setup concluído! Período: {data_de_str} até hoje ({data_ate_str})")

In [ ]:
# ==========================================
# Célula 2 - Inicialização do Chrome e Login com Persistência
# ==========================================
# Pasta local para salvar o perfil do Chrome e manter o login ativo nas próximas execuções
pasta_perfil_chrome = os.path.join(caminho_base, "chrome_profile")

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-popup-blocking")
chrome_options.add_argument("--disable-features=DownloadBubble,DownloadBubbleV2")
chrome_options.add_argument(f"--user-data-dir={pasta_perfil_chrome}")

prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_settings.popups": 0
}
chrome_options.add_experimental_option("prefs", prefs)

print("🚀 Iniciando navegador do Selenium com perfil persistente...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

wait_longo = WebDriverWait(driver, 600)  # Aguarda login manual se necessário
wait_curto = WebDriverWait(driver, 15)   # Esperas explícitas rápidas

driver.get(URL_LOGIN)
print("⏳ Verificando sessão ativa ou aguardando login no MKTZap...")

try:
    wait_longo.until(EC.url_contains("/chat"))
    print("✅ Login confirmado e sessão pronta para uso!")
except Exception as e:
    print(f"⚠️ Atenção ao verificar login: {e}")

In [ ]:
# ==========================================
# Célula 3 - Funções Auxiliares (CDP, Troca de Empresa e Downloads)
# ==========================================
def definir_pasta_download(pasta_temp):
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": pasta_temp
    })

def trocar_empresa(nome_empresa):
    print(f"🏢 Alternando para a empresa: '{nome_empresa}'...")

    # Redireciona para o /chat antes de trocar a empresa para garantir menu lateral visível
    if "/chat" not in driver.current_url:
        driver.get(URL_CHAT)
        wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))

    menu_user = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))
    driver.execute_script("arguments[0].scrollIntoView(true);", menu_user)
    driver.execute_script("arguments[0].click();", menu_user)

    btn_alterar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-change-company']")))
    driver.execute_script("arguments[0].click();", btn_alterar)

    input_empresa = wait_curto.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-modal-change-company-text-name']")))
    input_empresa.clear()

    for letra in nome_empresa:
        input_empresa.send_keys(letra)
        time.sleep(0.02)

    item_lista = wait_curto.until(EC.element_to_be_clickable((By.XPATH, f"//li[contains(text(), '{nome_empresa}')]")))
    driver.execute_script("arguments[0].click();", item_lista)

    btn_trocar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-botton-modal-change-company-button-ok']")))
    driver.execute_script("arguments[0].click();", btn_trocar)

    # Espera explícita para o recarregamento do painel
    time.sleep(4)

def aguardar_e_mover_download(pasta_temp, pasta_destino, prefixo_arquivo):
    inicio_espera = time.time()
    arquivo_baixado = None

    while time.time() - inicio_espera < 180: # Limite de 3 minutos
        arquivos = [f for f in glob.glob(os.path.join(pasta_temp, "*.*")) if not f.endswith('.crdownload') and not f.endswith('.tmp')]
        if arquivos:
            arquivo_baixado = max(arquivos, key=os.path.getmtime)
            if os.path.getmtime(arquivo_baixado) >= inicio_espera - 10:
                break
        time.sleep(2)

    if arquivo_baixado:
        ext = arquivo_baixado.split('.')[-1]
        nome_final = f"{prefixo_arquivo}_{primeiro_dia.strftime('%Y%m%d')}_a_{hoje.strftime('%Y%m%d')}.{ext}"
        caminho_final = os.path.join(pasta_destino, nome_final)

        if os.path.exists(caminho_final):
            os.remove(caminho_final)

        shutil.move(arquivo_baixado, caminho_final)
        print(f"  -> ✅ Arquivo salvo em: {nome_final}")
        return True
    else:
        print(f"  -> ⚠️ Timeout: O arquivo do relatório não apareceu na pasta de downloads.")
        return False

def baixar_do_background(t, prefixo_arquivo, nome_relatorio, limite_seg=300):
    """Abre a fila de Relatórios em Background, aguarda 'Finalizado' na 1ª linha e baixa o arquivo."""
    print("  -> Redirecionando para a fila de Relatórios em Background...")
    driver.get(URL_BACKGROUND)
    print("  -> Aguardando processamento da plataforma (Status: Finalizado)...")

    inicio_espera = time.time()
    while time.time() - inicio_espera < limite_seg:
        try:
            primeira_linha = wait_curto.until(EC.presence_of_element_located((By.XPATH, "//table//tbody//tr[1]")))
            if "Finalizado" in primeira_linha.text:
                print("  -> ✅ Status 'Finalizado' detectado! Iniciando download...")
                btn_dl = primeira_linha.find_element(By.XPATH, ".//button[@title='Download' or contains(@class, 'btn-outline-primary')]")
                driver.execute_script("arguments[0].click();", btn_dl)
                return aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], prefixo_arquivo)
        except Exception:
            pass
        time.sleep(4)
        driver.refresh()

    print(f"  -> ⚠️ Timeout atingido ({limite_seg // 60} min) aguardando finalização no background.")
    registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": nome_relatorio, "status": "Erro / Timeout Background"})
    return False

print("✅ Funções auxiliares carregadas com esperas explícitas!")

In [ ]:
# ==========================================
# Célula 4 - Extração do Relatório Analítico (Background via CSV)
# ==========================================
texto_atalho_data = "Este mês"  # 1º dia do mês de hoje até agora

for t in tenants:
    print(f"\n" + "="*50)
    print(f"🚀 INICIANDO EXTRAÇÃO ANALÍTICO: {t['nome_exibicao']}")
    print(f"   -> Empresa MKTZap: {t['nome_empresa_mktzap']}")
    print(f"   -> Destino Final:  {t['pasta_destino']}")
    print("="*50)

    try:
        definir_pasta_download(t["pasta_temp"])
        trocar_empresa(t["nome_empresa_mktzap"])

        driver.get(URL_ANALITICO)

        # 1. Clica no atalho de data
        print(f"  -> Clicando no atalho de data: '{texto_atalho_data}'...")
        opcao_data = wait_curto.until(
            EC.element_to_be_clickable((By.XPATH, f"//a[contains(text(), '{texto_atalho_data}')]"))
        )
        driver.execute_script("arguments[0].click();", opcao_data)

        # 2. Garante seleção do Formato CSV
        try:
            select_export = Select(wait_curto.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "select[ng-model='vm.exportAs']"))
            ))
            select_export.select_by_value("csv")
            print("  -> Formato CSV selecionado com sucesso.")
        except Exception as e_format:
            print(f"  -> Aviso ao selecionar formato CSV: {e_format}")

        # 3. Clica em "Gerar Relatório"
        print("  -> Clicando em 'Gerar Relatório' e enviando para background...")
        btn_gerar = wait_curto.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_analytic_report_generate']"))
        )
        driver.execute_script("arguments[0].click();", btn_gerar)

        # Confirma modal de segundo plano, se exibido
        time.sleep(1.5)
        modais = driver.find_elements(By.XPATH, "//button[contains(text(),'Confirmar')]")
        if modais and modais[0].is_displayed():
            driver.execute_script("arguments[0].click();", modais[0])

        # 4 e 5. Fila de background -> "Finalizado" -> download
        baixar_do_background(t, "bruto_analitico", "Analítico")

    except Exception as e:
        print(f"❌ Erro crítico na extração do Analítico ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": "Analítico", "status": f"Erro: {str(e)}"})

In [ ]:
## ==========================================
## Célula 5 - Extração do Relatório Completo
## ==========================================
#URL_COMPLETO = "https://EMPRESA.mktzap.com.br/report-bi/report-complete"
#
#for t in tenants:
#    print(f"\n==========================================")
#    print(f"📊 Processando Completo: {t['nome_exibicao']}")
#    print(f"==========================================")
#    
#    try:
#        definir_pasta_download(t["pasta_temp"])
#        trocar_empresa(t["nome_empresa_mktzap"])
#        
#        driver.get(URL_COMPLETO)
#        
#        input_from = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
#        input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
#        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, data_de_str)
#        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, data_ate_str)
#        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
#        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
#        
#        btn_gerar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[ga-event*='report_generate']")))
#        driver.execute_script("arguments[0].click();", btn_gerar)
#        
#        aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_completo")
#        
#    except Exception as e:
#        print(f"❌ Erro na extração do Completo ({t['nome_exibicao']}): {e}")
#        registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": "Completo", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 6 - Extração do Relatório de Enquetes (Injeção e Background)
# ==========================================
# Enquete alvo por tenant (comparação em minúsculas, por trecho do nome)
ENQUETE_ALVO = {
    "A": "pesquisa tenant a",
    "B": "pesquisa tenant b",
    "C": "pesquisa tenant c",
}

for t in tenants:
    print(f"\n==========================================")
    print(f"📊 Processando Enquetes: {t['nome_exibicao']}")
    print(f"==========================================")

    try:
        definir_pasta_download(t["pasta_temp"])
        trocar_empresa(t["nome_empresa_mktzap"])

        driver.get(URL_ENQUETES)
        time.sleep(2)

        # 1. Localiza os campos de data
        input_from = wait_curto.until(EC.presence_of_element_located((By.ID, "polls-from")))
        input_to = driver.find_element(By.ID, "polls-to")

        print(f"  -> Injetando período ({data_de_iso} até {data_ate_iso})...")

        # Camada A: Digitação nativa via Keys
        try:
            input_from.click()
            input_from.send_keys(Keys.CONTROL + "a")
            input_from.send_keys(data_de_keys)

            input_to.click()
            input_to.send_keys(Keys.CONTROL + "a")
            input_to.send_keys(data_ate_keys)
        except Exception:
            pass

        # Camada B e C: Injeção no DOM + Controlador ngModel do Angular
        driver.execute_script("""
            var elFrom = arguments[0];
            var elTo = arguments[1];
            var valFrom = arguments[2];
            var valTo = arguments[3];

            function forcarInputAngular(el, val) {
                el.value = val;
                el.setAttribute('value', val);

                if (typeof angular !== 'undefined') {
                    var $el = angular.element(el);
                    var ngModel = $el.controller('ngModel');
                    if (ngModel) {
                        ngModel.$setViewValue(val);
                        ngModel.$render();
                    }
                    $el.triggerHandler('input');
                    $el.triggerHandler('change');
                }
            }

            forcarInputAngular(elFrom, valFrom);
            forcarInputAngular(elTo, valTo);
        """, input_from, input_to, data_de_iso, data_ate_iso)

        time.sleep(1)

        val_f = input_from.get_attribute("value")
        val_t = input_to.get_attribute("value")
        print(f"  -> Confirmado nos campos: De [{val_f}] Até [{val_t}]")

        # 2. Aguarda carregamento das opções no dropdown #polls-poll
        print("  -> Aguardando carregamento das enquetes no dropdown...")
        wait_curto.until(lambda d: len(Select(d.find_element(By.ID, "polls-poll")).options) > 1)

        dropdown_poll = Select(driver.find_element(By.ID, "polls-poll"))

        # 3. Mapeamento Exato da Enquete por Tenant
        texto_alvo = ENQUETE_ALVO.get(t["id"], "pesquisa")

        encontrado = False
        for opt in dropdown_poll.options:
            if texto_alvo in opt.text.lower():
                dropdown_poll.select_by_visible_text(opt.text)
                print(f"  -> Enquete '{opt.text}' selecionada com sucesso.")
                encontrado = True
                break

        if not encontrado:
            dropdown_poll.select_by_index(1)
            print(f"  -> ⚠️ Alvo não encontrado. Selecionada a primeira disponível: '{dropdown_poll.first_selected_option.text}'")

        time.sleep(1.5)

        # 4. Clica em "Baixar detalhes da enquete"
        print("  -> Clicando em 'Baixar detalhes da enquete'...")
        btn_baixar = wait_curto.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Baixar detalhes da enquete')]")))
        driver.execute_script("arguments[0].click();", btn_baixar)

        # 5. Tratamento de Envio para Background (Modal de Confirmação)
        time.sleep(2)
        modais = driver.find_elements(By.XPATH, "//button[contains(@class, 'btn-primary') and contains(., 'Confirmar')]")

        if modais and modais[0].is_displayed():
            print("  -> Modal detetado. Enviando para fila de Background...")
            driver.execute_script("arguments[0].click();", modais[0])

            print("  -> Aguardando 5 segundos para o servidor registar o pedido...")
            time.sleep(5) # Evita a Race Condition de ler o relatório do Tenant anterior

            baixar_do_background(t, "bruto_enquetes", "Enquetes")
        else:
            print("  -> Download direto detetado. A guardar ficheiro...")
            aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_enquetes")

    except Exception as e:
        print(f"❌ Erro na extração de Enquetes ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": "Enquetes", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 7 - Extração do Relatório de Tickets
# ==========================================
for t in tenants:
    print(f"\n==========================================")
    print(f"📊 Processando Tickets: {t['nome_exibicao']}")
    print(f"==========================================")
    
    try:
        definir_pasta_download(t["pasta_temp"])
        trocar_empresa(t["nome_empresa_mktzap"])
        
        driver.get(URL_TICKETS)
        
        input_from = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
        input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, data_de_str)
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, data_ate_str)
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
        
        btn_gerar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_history_tickets_report_generate']")))
        driver.execute_script("arguments[0].click();", btn_gerar)
        
        aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_tickets")
        
    except Exception as e:
        print(f"❌ Erro na extração dos Tickets ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": "Tickets", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 8 - Encerrar a Sessão do Selenium
# ==========================================
try:
    driver.quit()
    print("🔒 Sessão do Selenium finalizada com sucesso! Todos os arquivos brutos foram processados.")
except Exception as e:
    print(f"⚠️ Aviso ao encerrar o driver: {e}")

In [ ]:
# ==========================================
# Célula 9 - Transformação Camada Gold & Padronização de Schema
# ==========================================
# Esquema oficial de 48 colunas na ordem exata (Exclusivo para Tenant A)
COLUNAS_OFICIAIS_ANALITICO_A = [
    'Protocolo', 'Empresa', 'Nome', 'CPF/CNPJ', 'Número do Contrato',
    'Valor do Contrato', 'CODIGO', 'GV', 'FUNÇÃO', 'PEDIDO FINALIZADO',
    'CONTATO DE SOLUÇÃO', 'CHAMADO', 'Fornecedor', 'Qtd Atendimentos',
    'testeLis', 'Função do Sistema', 'Ciclo - Health Check',
    'Recebidos - Health Check', 'CATEGORIA 1 (WR TRANSP)',
    'CATEGORIA 2 (WR TRANSP)', 'SOLUÇÃO APLICADA', 'Origem Health Check?',
    'Está em Fechamento?', 'Código do Impactado', 'Ciclo Vigente',
    'SETOR', 'MERCADO', 'Resposta pedido duplicado', 'Dados bancarios',
    'E-mail', 'Data Contato', 'Responsável', 'Tempo até ser atendido',
    'Tempo do atendimento', 'Primeiro Tempo de Espera', 'Tempo Médio de Espera',
    'Tempo de Operação', 'Fone Cliente', 'Canal de Entrada', 'Setor Entrada',
    'Setor Fim', 'Qtde Mensagens', 'Status de Classificação',
    'Status de finalização', 'Encerrado em', 'Observações',
    'Campos adicionais Web Chat', 'Quantidade de Tickets'
]

# Esquema oficial de 45 colunas na ordem exata (Exclusivo para Tenant C / Analitico C)
COLUNAS_OFICIAIS_ANALITICO_C = [
    'Protocolo', 'Empresa', 'Nome', 'CPF/CNPJ', 'Número do Contrato',
    'Valor do Contrato', 'CODE', 'BM CODE', 'BC CODE', 'FUNCTION',
    'SOLUTION APPLIED', 'FINISHED ORDER', 'CHAMADO', 'TICKET',
    'FINISHED ORDER_', 'QUANTITY', 'SM', 'Nome completo (usuário):',
    'E-mail (usuário):', 'Colaborador ou terceirizado?', 'Área (usuário)',
    'País (usuário)', 'Colaborador/Terceirizado', 'Empresa:', 'Localidade:',
    'Nome.1', 'Email', 'E-mail', 'Data Contato', 'Responsável',
    'Tempo até ser atendido', 'Tempo do atendimento', 'Primeiro Tempo de Espera',
    'Tempo Médio de Espera', 'Tempo de Operação', 'Fone Cliente',
    'Canal de Entrada', 'Setor Entrada', 'Setor Fim', 'Qtde Mensagens',
    'Status de Classificação', 'Status de finalização', 'Encerrado em',
    'Observações', 'Campos adicionais Web Chat'
]

# Schema fixo do Analítico por tenant (os demais tenants mantêm o layout nativo)
SCHEMA_GOLD_ANALITICO = {"A": COLUNAS_OFICIAIS_ANALITICO_A, "C": COLUNAS_OFICIAIS_ANALITICO_C}

def aplicar_regra_qtd(val):
    if pd.isna(val):
        return "1"
    val_str = str(val).strip()
    if val_str == "" or val_str.lower() in ["none", "nan", "null"]:
        return "1"
    try:
        num = float(val_str.replace(',', '.'))
        if num <= 0 or num >= 100:
            return "1"
        return str(int(num))
    except (ValueError, TypeError):
        return "1"

def limpar_texto_gold(val):
    """Decodifica entidades HTML (&#227; -> ã, &quot; -> "), remove espaços e converte nulos em vazio."""
    if pd.isna(val):
        return ""
    val = str(val)
    if "&" in val and ";" in val:
        val = html.unescape(val)
    val = val.strip()
    return "" if val in ("nan", "None", "NaN") else val

def encontrar_linha_cabecalho_real(caminho):
    encodings = ['utf-8-sig', 'latin1', 'utf-8', 'utf-16']
    for enc in encodings:
        try:
            with open(caminho, 'r', encoding=enc) as f:
                for idx, line in enumerate(f):
                    line_lower = line.lower()
                    if 'protocolo' in line_lower or 'protocol' in line_lower or 'data contato' in line_lower:
                        return idx
        except Exception:
            pass
    return 0

def carregar_csv_flexivel(caminho, rel_tipo=""):
    if rel_tipo == "tickets":
        skip_count = 1
    else:
        skip_count = encontrar_linha_cabecalho_real(caminho)

    for sep in [';', ',', '\t']:
        for enc in ['utf-8-sig', 'latin1', 'utf-8', 'utf-16']:
            try:
                df = pd.read_csv(
                    caminho, skiprows=skip_count, sep=sep, encoding=enc,
                    on_bad_lines='skip', low_memory=False
                )
                if len(df.columns) > 3:
                    return df
            except Exception:
                pass

    return pd.read_csv(caminho, skiprows=skip_count, sep=None, engine='python', encoding='utf-8-sig')

relatorios_tipos = ["analitico", "completo", "enquetes", "tickets"]
relatorio_qualidade = []

for t in tenants:
    pasta_destino = t["pasta_destino"]
    nome_tenant_slug = t["nome_exibicao"].split("-")[-1].strip().lower().replace(" ", "_")

    for rel in relatorios_tipos:
        arquivos_brutos = glob.glob(os.path.join(pasta_destino, f"bruto_{rel}_*.*"))
        if not arquivos_brutos:
            continue
        caminho_bruto = max(arquivos_brutos, key=os.path.getmtime)
        print(f"⏳ Processando Camada Gold do {rel.capitalize()} ({t['nome_exibicao']})...")

        # Exportação vazia: o MKTZap gera só a linha de título quando não há registros no período
        with open(caminho_bruto, encoding="utf-8-sig", errors="ignore") as f_bruto:
            linhas_uteis = sum(1 for linha in f_bruto if linha.strip())
        if linhas_uteis <= 1:
            print("  -> ℹ️ Exportação sem registros no período. Nada a gerar.")
            relatorio_qualidade.append({
                "tenant": t["nome_exibicao"], "relatorio": rel.capitalize(),
                "linhas": 0, "colunas": 0, "ajustes_qtd": 0, "tamanho_kb": 0,
                "status": "Sem registros"
            })
            os.remove(caminho_bruto)
            continue

        try:
            df = carregar_csv_flexivel(caminho_bruto, rel)
            df.columns = df.columns.astype(str).str.strip()

            # 🔥 Padronização nativa da Tenant B (mantida por segurança)
            df.rename(columns={'PEDIDOS FINALIZADOS': 'Pedidos Finalizados'}, inplace=True)

            # 🔥 REGRA EXCLUSIVA Tenant C: Excluir coluna Telefone
            if t["id"] == "C":
                cols_telefone = [c for c in df.columns if 'telefone' in c.lower()]
                if cols_telefone:
                    df.drop(columns=cols_telefone, inplace=True)
                    print(f"  -> ✂️ Coluna(s) {cols_telefone} removida(s) com sucesso (Regra Tenant C).")

            # Limpeza única de todas as colunas de texto (HTML, espaços e nulos)
            for col in df.select_dtypes(include=['object', 'string']).columns:
                df[col] = df[col].map(limpar_texto_gold)
                if rel == "analitico":  # o export manual do Analítico troca quebras de linha por espaço
                    df[col] = df[col].str.replace(r"\r\n|\r|\n", " ", regex=True)

            qtd_ajustes = 0
            cols_qtd = [c for c in df.columns if 'qtd' in c.lower() and 'atend' in c.lower()]
            if cols_qtd:
                col_alvo = cols_qtd[0]
                s_raw = df[col_alvo].astype(str)
                s_tratar = s_raw.apply(aplicar_regra_qtd)
                qtd_ajustes = (s_raw != s_tratar).sum()
                df['Qtd Atendimentos'] = s_tratar
                if col_alvo != 'Qtd Atendimentos':
                    df.drop(columns=[col_alvo], inplace=True, errors='ignore')

            # 🔥 APLICAÇÃO DO SCHEMA OFICIAL (TENANT A E Tenant C)
            colunas_oficiais = SCHEMA_GOLD_ANALITICO.get(t["id"]) if rel == "analitico" else None
            if colunas_oficiais:
                df = df.reindex(columns=colunas_oficiais, fill_value="").copy()
                print(f"  -> 🧩 Schema Oficial aplicado: {len(colunas_oficiais)} colunas ordenadas e sincronizadas.")

            nome_gold = f"{hoje.strftime('%Y-%m-%d')}-{nome_tenant_slug}-{rel}-periodo-{primeiro_dia.strftime('%Y-%m-%d')}-a-{hoje.strftime('%Y-%m-%d')}.csv"
            caminho_gold = os.path.join(pasta_destino, nome_gold)

            df.to_csv(caminho_gold, sep=';', index=False, encoding='utf-8-sig')

            tam_kb = round(os.path.getsize(caminho_gold) / 1024, 1)
            relatorio_qualidade.append({
                "tenant": t["nome_exibicao"],
                "relatorio": rel.capitalize(),
                "linhas": len(df),
                "colunas": len(df.columns),
                "ajustes_qtd": qtd_ajustes,
                "tamanho_kb": tam_kb,
                "status": "Sucesso"
            })

            os.remove(caminho_bruto)
            print(f"  -> ✅ Relatório gerado com sucesso: {nome_gold}")

        except Exception as e:
            print(f"  -> ❌ Erro ao converter {rel.capitalize()}: {str(e)}")
            relatorio_qualidade.append({
                "tenant": t["nome_exibicao"],
                "relatorio": rel.capitalize(),
                "linhas": 0, "colunas": 0, "ajustes_qtd": 0, "tamanho_kb": 0,
                "status": f"Erro Gold: {str(e)}"
            })

print("\n" + "="*70)
print("📊 PAINEL DE AUDITORIA E QUALIDADE DOS DADOS GOLD (DATA QUALITY REPORT)")
print("="*70)

if relatorio_qualidade:
    display(pd.DataFrame(relatorio_qualidade))
else:
    print("⚠️ Nenhum arquivo foi processado na Camada Gold.")

if registro_execucao:
    print("\n⚠️ FALHAS REGISTRADAS NA EXTRAÇÃO (SELENIUM):")
    display(pd.DataFrame(registro_execucao))

In [ ]:
# ==========================================
# Célula 10 - Limpeza das Pastas Temporárias do Sistema Operacional
# ==========================================
time.sleep(2)

def forcar_exclusao(func, path, exc_info):
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"⚠️ Aviso ao excluir {path}: {e}")

# Usa as pastas temporárias declaradas nos tenants (inclui Tenant C)
pastas_para_remover = [t["pasta_temp"] for t in tenants]

for caminho_pasta in pastas_para_remover:
    if os.path.exists(caminho_pasta):
        try:
            shutil.rmtree(caminho_pasta, onerror=forcar_exclusao)
            print(f"✅ Pasta limpa: {caminho_pasta}")
        except Exception as e:
            print(f"⚠️ Pasta trancada no SO/OneDrive: {e}")

print("✨ Pipeline completo finalizado com sucesso!")

In [ ]:
# ==========================================
# Célula 11 - Motor de Consolidação (Schema, Tipos e Gravação In-Place)
# ==========================================
# Todas as consolidações (Analítico, Enquetes e Tickets) passam por aqui.
# Regra de ouro: o Excel recebe TIPOS nativos (datetime / inteiro / texto).
# O formato "DD/MM/YYYY HH:MM" é apenas máscara de exibição da célula (number_format),
# nunca uma conversão do valor para texto.
# Depende da Célula 1 (imports, janela do mês, caminho_base e pasta_mes).
PASTA_AUTO = os.getenv("PASTA_SAIDA", os.path.join(os.getcwd(), "saida"))
FORMATO_PADRAO_DATA = "DD/MM/YYYY HH:MM"   # usado só se o arquivo de origem não tiver formato próprio
FORMATO_PADRAO_INTEIRO = "0"

# Formatos aceitos, em ordem. Formato explícito = dia e mês nunca são invertidos.
FORMATOS_DATA = [
    "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y",
    "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d",
]


# ==========================================
# CONVERSORES DE TIPO
# ==========================================
def _texto_para_datetime(valor):
    """Converte um valor isolado para Timestamp; devolve None se não for data."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    if isinstance(valor, (pd.Timestamp, dt.datetime)):
        return pd.Timestamp(valor)
    if isinstance(valor, dt.date):
        return pd.Timestamp(valor)
    if isinstance(valor, (int, float)) and not isinstance(valor, bool) and 20000 < valor < 80000:
        return (pd.Timestamp("1899-12-30") + pd.to_timedelta(valor, unit="D")).round("s")  # serial do Excel
    s = str(valor).strip()
    if s == "":
        return None
    s = s.split(".")[0] if s.count(":") == 2 and "." in s else s  # remove milissegundos
    for fmt in FORMATOS_DATA:
        try:
            return pd.Timestamp(datetime.strptime(s, fmt))
        except ValueError:
            continue
    return None


def para_datetime(serie):
    """Coluna 100% data: o que não for data vira vazio (NaT)."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie
    return pd.to_datetime(serie.map(_texto_para_datetime), errors="coerce")


def para_datetime_ou_texto(serie):
    """Coluna mista (ex.: 'Respondido em' = data ou 'Não respondido'): datas viram datetime, textos ficam como texto."""
    def conv(v):
        d = _texto_para_datetime(v)
        if d is not None:
            return d.to_pydatetime()
        return para_texto_valor(v)
    return serie.map(conv).astype(object)


def para_inteiro(serie):
    return pd.to_numeric(serie, errors="coerce").round().astype("Int64")


def corrigir_mojibake(texto):
    """Desfaz UTF-8 lido como Latin-1/CP1252 ('CLASSIFICAÃ‡ÃƒO' -> 'CLASSIFICAÇÃO').
    Textos corretos ('NÃO') falham no decode e voltam intactos."""
    if isinstance(texto, str) and ("Ã" in texto or "Â" in texto):
        try:
            return texto.encode("cp1252").decode("utf-8")
        except UnicodeError:
            return texto
    return texto


def para_texto_valor(v):
    """Texto limpo; números inteiros não ganham '.0'. Célula vazia continua vazia e texto '' continua ''
    (as bases de produção distinguem os dois)."""
    if v is None or v is pd.NaT or v is pd.NA or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    if isinstance(v, (pd.Timestamp, dt.datetime)):
        return v.strftime("%d/%m/%Y %H:%M:%S")
    return corrigir_mojibake(str(v)).strip()


def para_duracao(v):
    """Durações ('Tempo de Operação' etc.) ficam como texto HH:MM:SS, como o Tableau espera (SPLIT por ':')."""
    if isinstance(v, dt.time):
        return v.strftime("%H:%M:%S")
    if isinstance(v, (pd.Timedelta, dt.timedelta)):
        seg = int(pd.Timedelta(v).total_seconds())
        return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"
    if isinstance(v, float) and not pd.isna(v) and 0 <= v < 100:  # fração de dia do Excel
        seg = int(round(v * 86400))
        return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"
    return para_texto_valor(v)


def aplicar_schema(df, schema):
    """Aplica os tipos oficiais. Colunas fora do schema são tratadas como texto."""
    for col in df.columns:
        if col in schema.get("datetime", []):
            df[col] = para_datetime(df[col])
        elif col in schema.get("datetime_ou_texto", []):
            df[col] = para_datetime_ou_texto(df[col])
        elif col in schema.get("int", []):
            df[col] = para_inteiro(df[col])
        elif col.lower().startswith("tempo"):
            df[col] = df[col].map(para_duracao).astype(object)
        else:
            df[col] = df[col].map(para_texto_valor).astype(object)
    return df


# ==========================================
# LEITURA E GRAVAÇÃO
# ==========================================
def ler_historico_excel(caminho):
    """Lê o Excel célula a célula (openpyxl, somente leitura), sem o pandas adivinhar tipos nem
    converter textos como 'n/a' em nulo. Cabeçalhos repetidos ganham sufixo '.1', '.2'..."""
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    nome_aba = ws.title
    linhas = ws.iter_rows(values_only=True)
    cabecalho = list(next(linhas))
    while cabecalho and cabecalho[-1] is None:  # colunas fantasma sem cabeçalho
        cabecalho.pop()
    n = len(cabecalho)
    dados = [list(r[:n]) + [None] * (n - len(r[:n])) for r in linhas if any(v is not None for v in r[:n])]
    wb.close()

    colunas, vistos = [], {}
    for c in cabecalho:
        nome = corrigir_mojibake(str(c)).strip() if c is not None else "Unnamed"
        if nome in vistos:
            vistos[nome] += 1
            nome = f"{nome}.{vistos[nome]}"
        else:
            vistos[nome] = 0
        colunas.append(nome)
    return pd.DataFrame(dados, columns=colunas, dtype=object), nome_aba


def ler_layout_excel(caminho, max_linhas=500):
    """Lê (somente leitura) o cabeçalho exato e o formato de exibição de cada coluna do arquivo de origem."""
    wb = openpyxl.load_workbook(caminho, read_only=True)
    ws = wb.worksheets[0]
    linhas = ws.iter_rows(max_row=max_linhas + 1)
    cabecalho = [c.value for c in next(linhas)]
    while cabecalho and cabecalho[-1] is None:
        cabecalho.pop()
    formatos = {}
    for row in linhas:
        for i, c in enumerate(row[:len(cabecalho)]):
            if i not in formatos and isinstance(c.value, (dt.datetime, int, float)) and not isinstance(c.value, bool):
                formatos[i] = c.number_format
    wb.close()
    return {"cabecalho": cabecalho, "formatos": formatos}


def _cabecalho_saida(colunas, layout):
    """Reaproveita a grafia original do cabeçalho (ex.: 'ResponsÃ¡vel', 'Nome' duplicado no lugar de 'Nome.1')."""
    original = (layout or {}).get("cabecalho", [])
    if len(original) != len(colunas):
        return list(colunas)
    saida = []
    for orig, col in zip(original, colunas):
        base = corrigir_mojibake(str(orig)).strip() if orig is not None else None
        if base is not None and (col == base or re.fullmatch(re.escape(base) + r"\.\d+", col)):
            saida.append(orig)
        else:
            saida.append(col)
    return saida


def gravar_excel_in_place(df, caminho, nome_aba, schema, layout=None):
    """Substitui apenas a aba (mantém o arquivo e o File ID do Drive), herdando cabeçalho e formatos do layout."""
    layout = layout or {}
    formatos_orig = {}
    if len(layout.get("cabecalho", [])) == len(df.columns):
        formatos_orig = {df.columns[i]: f for i, f in layout.get("formatos", {}).items()}

    # Textos iniciados por '=' seriam gravados como fórmula (e perdidos): guarda as posições para forçar texto
    pos_formula = []
    for j, col in enumerate(df.columns):
        if df[col].dtype == object:
            mask = df[col].map(lambda v: isinstance(v, str) and v.startswith("="))
            pos_formula += [(i, j) for i in df.index[mask]]

    with pd.ExcelWriter(caminho, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df.to_excel(writer, sheet_name=nome_aba, index=False)
        ws = writer.sheets[nome_aba]

        for j, nome in enumerate(_cabecalho_saida(df.columns, layout), start=1):
            ws.cell(row=1, column=j).value = nome

        for i, j in pos_formula:
            cell = ws.cell(row=i + 2, column=j + 1)
            cell.value = df.iat[i, j]
            cell.data_type = "s"

        cols_data = set(schema.get("datetime", [])) | set(schema.get("datetime_ou_texto", []))
        cols_int = set(schema.get("int", []))
        for idx, col in enumerate(df.columns, start=1):
            if col in cols_data:
                fmt, tipo = formatos_orig.get(col, FORMATO_PADRAO_DATA), dt.datetime
            elif col in cols_int:
                fmt, tipo = formatos_orig.get(col, FORMATO_PADRAO_INTEIRO), int
            else:
                continue
            for (cell,) in ws.iter_rows(min_row=2, max_row=len(df) + 1, min_col=idx, max_col=idx):
                if isinstance(cell.value, tipo):
                    cell.number_format = fmt


# ==========================================
# MOTOR GENÉRICO DE CONSOLIDAÇÃO
# ==========================================
def consolidar(cfg, caminho_historico=None, caminho_saida=None):
    """
    cfg: nome, slug, relatorio, caminho_excel, schema, col_data, chave_unica (opcional),
         filtrar_inicio_gold (bool, default True).
    caminho_historico: lê o histórico de outro arquivo (ex.: '... old.xlsx' no reparo único).
    caminho_saida: grava em outro arquivo (testes). Padrão = caminho_excel (in-place).
    """
    print("\n" + "=" * 70)
    print(f"📊 CONSOLIDANDO {cfg['relatorio'].upper()}: {cfg['nome']}")
    print("=" * 70)

    caminho_excel = cfg["caminho_excel"]
    origem_hist = caminho_historico or caminho_excel
    destino = caminho_saida or caminho_excel
    schema = cfg["schema"]
    col_data = cfg["col_data"]

    pasta_gold = os.path.join(caminho_base, "documentos", pasta_mes, cfg["nome"])
    arquivos_gold = glob.glob(os.path.join(pasta_gold, f"*-{cfg['slug']}-{cfg['relatorio']}-*.csv"))

    if not os.path.exists(origem_hist):
        print(f"❌ Histórico não encontrado:\n   {origem_hist}")
        return None
    if not os.path.exists(destino):
        print(f"❌ Arquivo de destino não encontrado:\n   {destino}")
        return None
    if not arquivos_gold:
        print(f"❌ Nenhum Gold de {cfg['relatorio']} em:\n   {pasta_gold}")
        return None

    caminho_gold = max(arquivos_gold, key=os.path.getmtime)
    print(f"  -> Gold:      {os.path.basename(caminho_gold)}")
    print(f"  -> Histórico: {os.path.basename(origem_hist)}")

    # Leitura (layout = cabeçalho e formatos exatos do arquivo de origem)
    layout = ler_layout_excel(origem_hist)
    df_hist, nome_aba = ler_historico_excel(origem_hist)
    df_gold = pd.read_csv(caminho_gold, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    df_gold.columns = df_gold.columns.astype(str).str.strip()
    df_gold = df_gold.replace({"": None})

    # Nomes do Gold seguem a grafia do histórico (case-insensitive)
    mapa_cols = {c.lower(): c for c in df_hist.columns}
    df_gold.rename(columns=lambda c: mapa_cols.get(c.lower(), c), inplace=True)

    total_inicial = len(df_hist)

    # Tipagem ANTES de qualquer filtro: datas comparadas como datas, nunca como texto
    df_hist = aplicar_schema(df_hist, schema)
    df_gold = aplicar_schema(df_gold, schema)

    # Guilhotina: remove do histórico tudo do mês vigente em diante
    data_hist = para_datetime(df_hist[col_data])
    df_hist = df_hist[(data_hist < dt_inicio_filtro) | data_hist.isna()].copy()
    removidas = total_inicial - len(df_hist)
    print(f"  -> 🗑️ Removidas do histórico (>= {primeiro_dia:%d/%m/%Y}): {removidas:,}")

    # Gold: nada do dia de hoje é descartado; só o piso do 1º dia do mês (quando aplicável)
    if cfg.get("filtrar_inicio_gold", True):
        data_gold = para_datetime(df_gold[col_data])
        mascara = data_gold >= dt_inicio_filtro
        cortadas = int((~mascara).sum())
        df_gold = df_gold[mascara].copy()
        if cortadas:
            print(f"  -> ✂️ Descartadas do Gold (anteriores a {primeiro_dia:%d/%m/%Y}): {cortadas:,}")

    # Append
    df_final = pd.concat([df_hist, df_gold], ignore_index=True)

    # Deduplicação por chave (Gold prevalece por ser o dado mais recente)
    if cfg.get("chave_unica"):
        antes = len(df_final)
        df_final = df_final.drop_duplicates(subset=cfg["chave_unica"], keep="last").reset_index(drop=True)
        if antes != len(df_final):
            print(f"  -> 🔁 Duplicatas removidas por {cfg['chave_unica']}: {antes - len(df_final):,}")

    # Reaplica o schema após o concat (pandas pode ter mudado dtypes ao juntar)
    df_final = aplicar_schema(df_final, schema)

    print(f"💾 Gravando em {os.path.basename(destino)} (aba '{nome_aba}')...")
    gravar_excel_in_place(df_final.reset_index(drop=True), destino, nome_aba, schema, layout)

    resumo = {
        "tenant": cfg["nome"], "relatorio": cfg["relatorio"],
        "linhas_antes": total_inicial, "expurgadas": removidas,
        "adicionadas": len(df_gold), "linhas_final": len(df_final),
    }
    print(f"✅ {cfg['nome']}: {total_inicial:,} -> {len(df_final):,} linhas (+{len(df_gold):,} Gold)")
    del df_hist, df_gold, df_final
    gc.collect()
    return resumo


# ==========================================
# SCHEMAS OFICIAIS (tipos esperados pelo Tableau Prep)
# ==========================================
SCHEMA_ANALITICO_A = {
    "datetime": ["Data Contato", "Encerrado em"],
    "int": ["Qtd Atendimentos", "Qtde Mensagens", "Quantidade de Tickets"],
}
SCHEMA_ANALITICO_B = {
    "datetime": ["Data Contato", "Encerrado em"],
    "int": ["Cant. de Atendimientos", "Qtde Mensagens", "Quantidade de Tickets"],
}
SCHEMA_ANALITICO_C = {
    "datetime": ["Data Contato", "Encerrado em"],
    "int": ["Qtde Mensagens"],
}
SCHEMA_ENQUETES = {
    "datetime": ["Data do Contato", "Enviado em"],
    "datetime_ou_texto": ["Respondido em"],  # data ou "Não respondido"
}
SCHEMA_TICKETS_A = {
    "datetime": ["Data de Criação do Atendimento", "Data de Finalização",
                 "Data de Criação do Ticket", "Data de Conversão do Atendimento"],
    "int": ["Qtd Atendimentos", "Ciclo Vigente"],
}
SCHEMA_TICKETS_B = {
    # 'Data de Conversão do Atendimento' fica como texto: é string no fluxo do Tableau
    "datetime": ["Data de Criação do Atendimento", "Data de Finalização", "Data de Criação do Ticket"],
    "int": ["Cant. de Atendimientos"],
}

CONFIGS_ANALITICO = [
    {"nome": "Tenant A", "slug": "tenant_a", "relatorio": "analitico",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant A", "Analitico_A.xlsx"),
     "schema": SCHEMA_ANALITICO_A, "col_data": "Data Contato"},
    {"nome": "Tenant B", "slug": "tenant_b", "relatorio": "analitico",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant B", "Analitico_B.xlsx"),
     "schema": SCHEMA_ANALITICO_B, "col_data": "Data Contato"},
    {"nome": "Tenant C", "slug": "tenant_c", "relatorio": "analitico",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant C", "Analitico_C.xlsx"),
     "schema": SCHEMA_ANALITICO_C, "col_data": "Data Contato"},
]

# Enquetes: o MKTZap filtra a exportação pela data de ENVIO (o Gold traz 'Data do Contato'
# de meses anteriores). Por isso guilhotina e piso do mês usam 'Enviado em'.
CONFIGS_ENQUETES = [
    {"nome": "Tenant A", "slug": "tenant_a", "relatorio": "enquetes",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant A", "Enquetes_A.xlsx"),
     "schema": SCHEMA_ENQUETES, "col_data": "Enviado em"},
    {"nome": "Tenant B", "slug": "tenant_b", "relatorio": "enquetes",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant B", "Enquetes_B.xlsx"),
     "schema": SCHEMA_ENQUETES, "col_data": "Enviado em"},
    {"nome": "Tenant C", "slug": "tenant_c", "relatorio": "enquetes",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant C", "Enquetes_C.xlsx"),
     "schema": SCHEMA_ENQUETES, "col_data": "Enviado em"},
]

# Tickets: guilhotina pela criação do atendimento, mas o Gold não recebe piso de data
# (um ticket de atendimento do mês anterior não se perde) e a chave 'ID do Ticket' evita duplicidade.
CONFIGS_TICKETS = [
    {"nome": "Tenant A", "slug": "tenant_a", "relatorio": "tickets",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant A", "Tickets_A.xlsx"),
     "schema": SCHEMA_TICKETS_A, "col_data": "Data de Criação do Atendimento",
     "chave_unica": ["ID do Ticket"], "filtrar_inicio_gold": False},
    {"nome": "Tenant B", "slug": "tenant_b", "relatorio": "tickets",
     "caminho_excel": os.path.join(PASTA_AUTO, "Tenant B", "Tickets_B.xlsx"),
     "schema": SCHEMA_TICKETS_B, "col_data": "Data de Criação do Atendimento",
     "chave_unica": ["ID do Ticket"], "filtrar_inicio_gold": False},
]

# Versões íntegras do histórico (cópias feitas pelo Windows: "<nome> - Copia.xlsx"), usadas no reparo e na auditoria
SUFIXO_COPIA = " - Copia"
ARQUIVOS_OLD = {
    nome: nome.replace(".xlsx", f"{SUFIXO_COPIA}.xlsx")
    for nome in [
        "Analitico_A.xlsx", "Analitico_B.xlsx", "Analitico_C.xlsx",
        "Enquetes_A.xlsx", "Enquetes_B.xlsx", "Enquetes_C.xlsx",
        "Tickets_A.xlsx", "Tickets_B.xlsx",
    ]
}

def caminho_old(cfg):
    pasta, nome = os.path.split(cfg["caminho_excel"])
    return os.path.join(pasta, ARQUIVOS_OLD[nome])

print("✅ Motor de consolidação carregado (schemas: Analítico, Enquetes e Tickets).")

In [ ]:
# ==========================================
# Célula 12 - Consolidação no Google Drive: Analítico (Tenant A, Tenant B e Tenant C)
# ==========================================
resumo_consolidacao = []
for cfg in CONFIGS_ANALITICO:
    r = consolidar(cfg)
    if r:
        resumo_consolidacao.append(r)

if resumo_consolidacao:
    display(pd.DataFrame(resumo_consolidacao))

In [ ]:
# ==========================================
# Célula 13 - Consolidação no Google Drive: Enquetes (Tenant A, Tenant B e Tenant C)
# ==========================================
# Guilhotina e piso do mês usam 'Enviado em' (o MKTZap filtra a exportação pela data de envio).
resumo_consolidacao = []
for cfg in CONFIGS_ENQUETES:
    r = consolidar(cfg)
    if r:
        resumo_consolidacao.append(r)

if resumo_consolidacao:
    display(pd.DataFrame(resumo_consolidacao))

In [ ]:
# ==========================================
# Célula 14 - Consolidação no Google Drive: Tickets (Tenant A e Tenant B)
# ==========================================
# Datas gravadas como datetime e 'Cant. de Atendimientos'/'Qtd Atendimentos' como inteiro.
# Deduplicação por 'ID do Ticket' (o registro do Gold prevalece).
resumo_consolidacao = []
for cfg in CONFIGS_TICKETS:
    r = consolidar(cfg)
    if r:
        resumo_consolidacao.append(r)

if resumo_consolidacao:
    display(pd.DataFrame(resumo_consolidacao))

In [ ]:
# ==========================================
# Célula 15 - REPARO ÚNICO: Reconstrução do histórico a partir dos arquivos "old"
# ==========================================
# Executar UMA vez. As versões anteriores das Células 13/14 (e da 12 para o Tenant C) gravaram
# datas como texto e inverteram dia/mês (ex.: 05/10 -> 10/05), duplicando linhas.
# Esta célula relê o histórico íntegro dos arquivos "old" (somente leitura) e regrava
# o arquivo atual da pasta auto com os tipos corretos + o Gold mais recente.
# Os arquivos de produção (EXP BR, EXP OIS, Enquetes, Tenant C) NÃO são tocados.
EXECUTAR_REPARO = False  # mude para True, rode a célula uma vez e volte para False

# Todas as bases com cópia "old" (ARQUIVOS_OLD, Célula 11) são reconstruídas:
# o histórico vem do "old" (layout de produção) e o mês vigente vem do Gold.
BASES_A_REPARAR = list(ARQUIVOS_OLD)

if not EXECUTAR_REPARO:
    print("⏸️ Reparo desativado (EXECUTAR_REPARO = False).")
else:
    resumo_reparo = []
    for cfg in CONFIGS_ANALITICO + CONFIGS_ENQUETES + CONFIGS_TICKETS:
        if os.path.basename(cfg["caminho_excel"]) not in BASES_A_REPARAR:
            continue
        r = consolidar(cfg, caminho_historico=caminho_old(cfg))
        if r:
            resumo_reparo.append(r)
    if resumo_reparo:
        display(pd.DataFrame(resumo_reparo))

In [ ]:
# ==========================================
# Célula Auxiliar - Auditoria: arquivo "old" vs. arquivo atual (pasta auto)
# ==========================================
# Substitui as antigas células auxiliares de comparação (Analítico, Enquetes e Tickets).
# Compara volume total e contagem mensal do histórico anterior ao mês vigente.
EXECUTAR_AUDITORIA = False

if not EXECUTAR_AUDITORIA:
    print("⏸️ Auditoria desativada (EXECUTAR_AUDITORIA = False).")
else:
    for cfg in CONFIGS_ANALITICO + CONFIGS_ENQUETES + CONFIGS_TICKETS:
        caminho_novo = cfg["caminho_excel"]
        nome = os.path.basename(caminho_novo)
        caminho_hist_old = caminho_old(cfg)
        print("\n" + "=" * 70)
        print(f"📊 {cfg['relatorio'].upper()} - {cfg['nome']} ({nome})")
        print("=" * 70)
        if not (os.path.exists(caminho_hist_old) and os.path.exists(caminho_novo)):
            print(f"⚠️ Arquivo não encontrado: {caminho_hist_old if not os.path.exists(caminho_hist_old) else caminho_novo}")
            continue

        contagens = {}
        for tag, caminho in (("OLD", caminho_hist_old), ("NEW", caminho_novo)):
            df_aud, _ = ler_historico_excel(caminho)
            datas = para_datetime(df_aud[cfg["col_data"]])
            print(f"• {tag}: {len(df_aud):,} linhas | {os.path.getsize(caminho) / 1024:,.1f} KB | datas inválidas: {datas.isna().sum():,}")
            contagens[f"Qtd_{tag}"] = datas[datas < dt_inicio_filtro].dt.to_period("M").value_counts()

        df_meses = pd.DataFrame(contagens).fillna(0).astype(int)
        df_meses["Diferenca"] = df_meses["Qtd_NEW"] - df_meses["Qtd_OLD"]
        divergentes = df_meses[df_meses["Diferenca"] != 0]
        print(f"📅 Meses do histórico com divergência: {len(divergentes)}")
        if len(divergentes):
            display(divergentes.sort_index())